# AWS Bedrock AgentCore — LangGraph Agent

This notebook deploys a LangGraph ReAct agent to Bedrock AgentCore Runtime using the **HTTP protocol**, then runs concurrent tests to measure session management and cold/warm start performance — mirroring the MCP server experiments.

### Agent
The agent uses Claude (via Bedrock) with math tools (add, multiply) in sync and async variants. Each server instance generates a unique UUID to track session-to-VM mapping, identical to the MCP server pattern.

### Tests
- Invoke the agent in parallel with unique sessions
- Invoke the agent in parallel with a shared session
- Compare latency and server instance distribution against MCP results

## Install Dependencies

In [ ]:
#!pip install --force-reinstall -U -r requirements.txt --quiet

## Setup

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

agent_name = "blog_langgraph_agent"

print(f"Using AWS region: {region}")

## Deploy AgentCore LangGraph Agent

Configure and launch the LangGraph agent using the **HTTP** protocol (unlike MCP for the math server). The agent exposes a `POST /invocations` endpoint that accepts `{"prompt": "...", "tool_mode": "sync|async"}` payloads.

In [ ]:
# Configure and launch the LangGraph agent
agentcore_runtime = Runtime()
response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="HTTP",
    agent_name=agent_name,
)

launch_result = agentcore_runtime.launch()
print(f"Launch completed. Agent ARN: {launch_result.agent_arn}")

# Store the agent ARN in Parameter Store for the test script
ssm_client.put_parameter(
    Name="/blog_langgraph_agent/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for blog_langgraph_agent",
    Overwrite=True,
)

## Test: Async Tool, 5 Calls, Unique Sessions

Each call gets its own session, so each is routed to a separate warm pool instance. Expect ~5 unique server IDs and no cold starts (warm pool has 10 instances).

In [ ]:
!python test_agent.py async --calls 5 --session unique

## Test: Async Tool, 15 Calls, Unique Sessions — Cold Start Beyond Warm Pool

With 15 concurrent requests exceeding the warm pool of 10:
- First 10 served immediately from the warm pool (~no cold start)
- Remaining 5 trigger cold starts with additional latency

In [ ]:
!python test_agent.py async --calls 15 --session unique

## Test: Async Tool, 5 Calls, Shared Session

All calls share the same session, routing to a single instance. With the LangGraph agent (which involves LLM inference), this tests whether the async event loop can handle concurrent requests on one VM.

In [ ]:
!python test_agent.py async --calls 5 --session shared

## Test: Sync Tool, 5 Calls, Unique Sessions

Sync tool variant with unique sessions. Each call blocks its thread with `time.sleep(2)`, but since each goes to a separate instance, they still run in parallel.

In [ ]:
!python test_agent.py sync --calls 5 --session unique

## Test: Sync Tool, 5 Calls, Shared Session — Sequential Bottleneck

Shared session pins all calls to one instance. Since the sync tool blocks with `time.sleep(2)`, requests are processed sequentially — expect significantly longer wall time.

In [ ]:
!python test_agent.py sync --calls 5 --session shared

## Analysis

Key differences from MCP server tests:
- **LLM inference overhead**: Each agent call involves a Bedrock Claude invocation, adding several seconds beyond the tool's `sleep(2)`. Total per-call latency will be higher than raw MCP tool calls.
- **Same session behavior**: Session-to-VM mapping is identical — shared sessions pin to one instance, unique sessions fan out across the warm pool.
- **Warm pool applies equally**: The 10-instance warm pool works the same for HTTP protocol agents as for MCP servers.

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# try:
#     ssm_client.delete_parameter(Name="/blog_langgraph_agent/runtime_iam/agent_arn")
#     print("Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("Parameter Store parameter not found")

# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=agent_name,
#     delete_ecr_repo=True,
# )